# Dynamic Pricing Env with Lag 

> Overarching DP env

In [ ]:
#| default_exp envs.pricing.dynamic_lag

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, Literal

from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss, quantile_loss
from ddopai.envs.pricing.base import BasePricingEnv
import gymnasium as gym

import numpy as np
import time

In [ ]:
# | export
class LagDynamicPricingEnv(BasePricingEnv):
    """
    Class implementing the dynamic pricing and learning problem for the single-item case.
    This version removes the SKU dimension from the observations and parameters.
    """
    def __init__(self,
        alpha: Union[np.ndarray, Parameter, int, float] = 1.0,  # market size
        beta: Union[np.ndarray, Parameter, int, float] = 0.5,   # price elasticity
        p_bound_low: Union[np.ndarray, Parameter, int, float] = 0.0,  # lower price bound
        p_bound_high: Union[np.ndarray, Parameter, int, float] = 1.0, # upper price bound
        dataloader: BaseDataLoader = None,  # dataloader TODO: replace with pricing dataloader
        num_SKUs: Union[np.ndarray, Parameter, int, float] = None,  # number of SKUs; ignored here
        gamma: float = 1,  # discount factor
        
        lag_window: int = 1,
        nb_features: int = 1,  # number of features
        covariance: Union[np.ndarray, Parameter, int, float] = 1,  # standard deviation of the features
        noise_std: Union[np.ndarray, Parameter, int, float] = 1,     # standard deviation of the noise
        function_form: Union[np.ndarray, Parameter, str] = "linear", # functional form of the demand function
        inv: Union[np.ndarray, Parameter, int, float] = 100,         # inventory
        horizon_train: int | str = "use_all_data",  # if "use_all_data", then horizon is inferred from the DataLoader
        postprocessors: list[object] | None = None,  # default is empty list 
        mode: str = "train", 
        return_truncation: str = False,  # TODO:Why is this a string?
        env_type: dict = {"inv": False, "reference_price": False},
        ) -> None:

        self.print = False

        if dataloader is None:
            dataloader = self.update_dataloader()
            
        # Ignore the multi-SKU feature: always use one SKU.
        num_SKUs = 1  
        # Remove SKU from parameters.
        # self.set_param("num_SKUs", num_SKUs, new=True)   <-- removed
        
        self.set_param("alpha", alpha, shape=np.atleast_1d(alpha).shape, new=True)
        self.set_param("beta", beta, shape=np.atleast_1d(beta).shape, new=True)
        # Force p_bound_low and p_bound_high to be 1D arrays of length 1.
        self.set_param("p_bound_low", p_bound_low, shape=(1,), new=True)
        self.set_param("p_bound_high", p_bound_high, shape=(1,), new=True)
        
        self.set_param("nb_features", nb_features, new=True)
        if isinstance(covariance, np.ndarray):
            self.set_param("covariance", covariance, shape=covariance.shape, new=True)
        else:
            self.set_param("covariance", covariance, new=True)
        if isinstance(noise_std, np.ndarray):
            self.set_param("noise_std", noise_std, shape=noise_std.shape, new=True)
        else:
            self.set_param("noise_std", noise_std, new=True)
        self.set_param("function_form", function_form, new=True)
        
        # Inventory parameters: use the first element.
        self.set_param("inv", inv, inv.shape, new=True)
        relative_inv = inv.copy()
        relative_inv[-1] = np.float32(1.0)  # np.float64(1.0)
        self.set_param("relative_inv", relative_inv, relative_inv.shape, new=True)
        self.set_param("inv_per_episode", inv, inv.shape, new=True)
        self.set_param("horizon_train", horizon_train, new=True)
        
        self.set_param("info_history", {}, new=True)
        self.set_param("lag_window", lag_window, new=True)
        self.set_param("lag_window_history", {}, new=True)
        
        # Initialize lag history for a single SKU.
        for i in range(int(lag_window)):
            self.lag_window_history[i] = {
                "action": np.zeros(1), 
                "reward": 0.0, 
                "done": 1.0
            }
            # For features, expect a vector of length nb_features.
            self.lag_window_history[i]["X"] = np.zeros((nb_features,))
            # For inventory, use a single value.
            self.lag_window_history[i]["inv"] = np.array([1])
            
        self.set_param("env_type", env_type, new=True)
        
        low = np.min(dataloader.X, axis=0)
        high = np.max(dataloader.X, axis=0)

        # Set the observation space without the SKU dimension.
        self.set_observation_space(dataloader.X_shape, feature_low=low, feature_high=high, lag_window=lag_window)
        self.set_action_space(dataloader.Y_shape, low=self.p_bound_low, high=self.p_bound_high)
        
        mdp_info = MDPInfo(self.observation_space, self.action_space, gamma=gamma, horizon=horizon_train)
        
        super().__init__(mdp_info=mdp_info,
                         postprocessors=postprocessors,
                         mode=mode, return_truncation=return_truncation,
                         dataloader=dataloader,
                         horizon_train=horizon_train)
    
    
    def set_observation_space(self, feature_shape, feature_low=-np.inf, feature_high=np.inf, lag_window=1, samples_dim_included=True):
        """
        Set the observation space of the environment.

        This version adds an extra time step (time_dim = lag_window + 1) to include current features
        and removes the SKU dimension.
        
        The final shapes will be:
        - features: (lag_window+1, nb_features)
        - inventory: (lag_window+1, 1)
        - actions: (lag_window+1, 1)
        - reward: (lag_window+1, 1)
        - done: (lag_window+1, 1)
        """
        spaces = {}
        time_dim = lag_window + 1  # Time dimension is the lag window plus the current step.
        
        if isinstance(feature_shape, tuple):
            if samples_dim_included:
                # Remove the sample dimension if present.
                feature_shape = feature_shape[1:]
            
            # Broadcast the low and high arrays to shape (time_dim, *feature_shape).
            low = np.broadcast_to(feature_low, (time_dim, *feature_shape))
            high = np.broadcast_to(feature_high, (time_dim, *feature_shape))

            spaces["features"] = gym.spaces.Box(
                low=low,
                high=high,
                shape=(time_dim, *feature_shape),
                dtype=np.float32
            )
        elif feature_shape is None:
            pass
        else:
            raise ValueError("Shape for features must be a tuple or None")

        # Define other spaces without a SKU dimension.
        spaces["inventory"] = gym.spaces.Box(
            low=0,
            high=1,
            shape=(time_dim, 1),
            dtype=np.float32
        )
        p_bound_low_broadcasted = np.broadcast_to(self.p_bound_low, (time_dim, 1))
        p_bound_high_broadcasted = np.broadcast_to(self.p_bound_high, (time_dim, 1))
        spaces["actions"] = gym.spaces.Box(
            low=p_bound_low_broadcasted,
            high=p_bound_high_broadcasted,
            shape=(time_dim, 1),
            dtype=np.float32
        )
        spaces["reward"] = gym.spaces.Box(
            low=0,
            high=np.inf,
            shape=(time_dim, 1),
            dtype=np.float32
        )
        spaces["done"] = gym.spaces.Box(
            low=0,
            high=1,
            shape=(time_dim, 1),
            dtype=np.float32
        )
        self.observation_space = gym.spaces.Dict(spaces)
        
        
    def step_(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, dict]:
        """
        Step function implementing the dynamic pricing and learning problem.
        
        This version assumes that the observation state has a time dimension of (lag_window + 1)
        and that the current values are in the last row of the 'features' key.
        The lag history is updated by storing the current values.
        """
        # Ensure action is a 1D array if provided as (1, n) array.
        if action.ndim == 2 and action.shape[0] == 1:
            action = np.squeeze(action, axis=0)

        terminated = False
        observation, reward_functions = self.get_observation()
        # Expect observation["features"] to have shape (lag_window+1, nb_features)
        X = observation["features"]#[0]
        
        # Since there is only one SKU, process directly.
        current_x = X[-1]  # current features are in the last time step
        demand, demand_noise_free = reward_functions[0](current_x, action)
        # Inventory handling (if enabled)
        if self.env_type["inv"]:
            if (demand / self.inv[0]) >= self.relative_inv[0]:
                demand = self.relative_inv[0] * self.inv[0]
                if (demand_noise_free / self.inv[0]) >= self.relative_inv[0]:
                    demand_noise_free = self.relative_inv[0] * self.inv[0]
                self.relative_inv[0] = 0
            else:
                self.relative_inv[0] -= demand / self.inv[0]
        
        reward = demand * action
        
        if self.relative_inv[0] == 0:
            terminated = True
            
        info = dict(
            inv = self.inv.copy() * self.relative_inv.copy(),
            demand = demand,
            demand_noise_free = demand_noise_free,
            action = action.copy(),
            reward = reward
        )
        
        self.info_history[len(self.info_history)] = info
        
        truncated = self.set_index()

        # Update lag history with the current timestamp's values.
        self.lag_window_history[len(self.lag_window_history)] = {
            "action": action,
            "reward": np.array(reward),
            "done": np.array([int(terminated or truncated)]),
            "X": X[-1, :],  # current features (last time step)
            "inv": np.array(self.relative_inv[0].copy())
        }

        if truncated:
            if self.mode in ["test", "val"]:
                observation = None
            else:
                observation, _ = self.get_observation()
            return observation, reward, terminated, truncated, info
        else:
            observation, _ = self.get_observation()
            if self.print:
                print("next_period:", self.index + 1)
                print("next observation:", observation)
                time.sleep(3)
            return observation, reward, terminated, truncated, info

            
    def get_info_history(self) -> dict:
        """ Return the history of the environment. """
        return self.info_history
    
    def reset(self,
        start_index: int | str = None,  # index to start from
        state: np.ndarray = None         # initial state
        ) -> Tuple[np.ndarray, bool]:
        """
        Reset function for the environment. Returns the first observation.
        """
        truncated = self.reset_index(start_index)
        
        self.info_history = {}
        self.lag_window_history = {}

        for i in range(int(self.lag_window)):
            self.lag_window_history[i] = {
                "action": -np.ones(1),
                "reward": -np.array([1]),
                "done": np.array([1]),
                "X": np.zeros((self.nb_features[0],)),  # shape: (nb_features,)
                "inv": np.array([1])
            }
            
        observation, demand = self.get_observation()
        return observation
     
    def get_observation(self):
        """
        Constructs a unified observation where each key has a time dimension of (lag_window+1).
        
        The final shapes are:
        - features: (lag_window+1, nb_features)
        - inventory: (lag_window+1, 1)
        - actions: (lag_window+1, 1)
        - rewards: (lag_window+1, 1)
        - done: (lag_window+1, 1)
        """
        # Retrieve current features and reward functions.
        x, reward_functions = self.dataloader[self.index]
        
        # Determine history indices for the last 'lag_window' entries.
        start_idx = int(self.index) - int(self.lag_window) + 1
        end_idx = int(self.index) + 1  # yields lag_window historical entries
        
        # Gather historical data.
        hist_actions = np.array([self.lag_window_history[i]["action"] for i in range(start_idx, end_idx)])
        hist_rewards = np.array([self.lag_window_history[i]["reward"] for i in range(start_idx, end_idx)])
        hist_done    = np.array([self.lag_window_history[i]["done"] for i in range(start_idx, end_idx)])
        hist_features = np.array([self.lag_window_history[i]["X"] for i in range(start_idx, end_idx)])
        hist_inv     = np.array([self.lag_window_history[i]["inv"] for i in range(start_idx, end_idx)])
        
        # Default current values.
        default_action = np.full((1,), -1.0)
        default_reward = np.full((1,), -1.0)
        default_done   = np.full((1,), 0.0)
        
        actions = np.concatenate([hist_actions, default_action[None, :]], axis=0)
        rewards = np.concatenate([hist_rewards, default_reward[None, :]], axis=0)
        done    = np.concatenate([hist_done, default_done[None, :]], axis=0)
        # Now, x[None, ...] will be of shape (1, nb_features)
        features = np.concatenate([hist_features, x], axis=0)
        inventory_seq = np.concatenate([hist_inv, np.array(self.relative_inv)], axis=0)
        
        #actions = actions[None, ...]  # Expand dimensions to (1, time_dim, x)
        #rewards = rewards[None, ...]  # Expand dimensions to (1, time_dim, x)
        #done    = done[None, ...]     # Expand dimensions to (1, time_dim, x)
        #features = features[None, ...]  # Expand dimensions to (1, time_dim, x)
        #inventory_seq = inventory_seq[None, ...]  # Expand dimensions to (1, time_dim, x)
        
        observation = {
            "features": features,         # (1, lag_window+1, nb_features)
            "inventory": inventory_seq,   # (1, lag_window+1, 1)
            "actions": actions,           # (1, lag_window+1, 1)
            "rewards": rewards,           # (1, lag_window+1, 1)
            "done": done,                 # (1, lag_window+1, 1)
        }
        return observation, reward_functions

    def reset_env(self, epoch):
        # Implementation omitted.
        pass
